# Proposed generator: generation from a trained model

Loads the model that `proposed_batch.ipynb` saved in `OUT_DIR/artifacts/<SOURCE_MODEL_KEY>/` and generates the test flows again, with flow lengths drawn uniformly with replacement from the lengths of the training flows (`FLOW_LEN_MODE = 'empirical'`). No training. Run by `scripts/run_matrix.py --plan genonly`.

In [ ]:
SEED             = 0
DATASET          = 'facebook_video'
RAW_PATH_STR     = 'data/facebook_video.csv'
OUT_DIR          = 'runs/facebook_video/seed0'
MODEL_KEY        = 'fl_empirical'   # name of the output file
TEST_SIZE        = 0.3              # not used: the split is read from split.json
SOURCE_MODEL_KEY = 'proposed'       # trained run whose artifacts are loaded
FLOW_LEN_MODE    = 'empirical'      # 'oracle' (test flows) or 'empirical' (drawn from the training flows)
TEMPERATURE_IDLE = None             # None: the value of the trained run

In [ ]:
import json
import os
import random
import sys
from pathlib import Path

import joblib
import numpy as np
import torch

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()   # run from the repository root
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'scripts'))
from data_utils import atomic_write_csv, load_packets, read_split
from hmm_mdn import GlobalStateLenMDN, flow_length_selector, generate_flows

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':16:8'
torch.use_deterministic_algorithms(True)
np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
df = load_packets(RAW_PATH_STR)
flow_sizes = df.groupby('flow_id').size()
train_flows, test_flows = read_split(Path(OUT_DIR) / 'split.json')

art = Path(OUT_DIR) / 'artifacts' / SOURCE_MODEL_KEY
meta = json.loads((art / 'meta.json').read_text())
hmm_boot, scaler_pkt, scaler_len = (joblib.load(art / f) for f in ('hmm.pkl', 'scaler_pkt.pkl', 'scaler_len.pkl'))
K_STATES = meta['K']
IDLE_K = K_STATES - 1 if meta['idle'] else -1
USE_FLOW_LEN = not meta['no_flow_len']
DIST_FAMILY = meta['dist_family']
if TEMPERATURE_IDLE is None:
    TEMPERATURE_IDLE = meta.get('temperature_idle', 1.0)
model_mdn = GlobalStateLenMDN(K_STATES, hidden=meta['H'], n_mix=meta['M'], use_flow_len=USE_FLOW_LEN).to(device)
model_mdn.load_state_dict(torch.load(art / 'mdn.pth', map_location=device, weights_only=True))
model_mdn.eval()

In [ ]:
flow_len_for = flow_length_selector(FLOW_LEN_MODE, flow_sizes, train_flows, SEED)
synth_df = generate_flows(model_mdn, hmm_boot.transmat_, hmm_boot.startprob_, scaler_pkt, scaler_len,
                          test_flows, flow_len_for, idle_state=IDLE_K, temp_idle=TEMPERATURE_IDLE,
                          dist_family=DIST_FAMILY, use_flow_len=USE_FLOW_LEN, seed=SEED, device=device)
atomic_write_csv(synth_df, Path(OUT_DIR) / f'{MODEL_KEY}.csv')
print(f'generated {len(synth_df):,} packets in {synth_df.flow_id.nunique()} flows ({FLOW_LEN_MODE} flow lengths)')